# Slowly Changing Dimension Example using Jinja2

## Prerequisites

1. Install jinja2 library on cluster. Create a requirements.txt file with content below and install library on cluster
 - jinja2

See jinja2 doc for more examples;
 - https://jinja.palletsprojects.com/en/stable/

In [1]:
%pip install jinja2 

In [6]:
%sql

--clean up
Drop table if exists lake.scd.dim_customer


OK

In [7]:
spark.sql("CREATE SCHEMA IF NOT EXISTS lake.scd")

DataFrame[]

## The Jinja2 template for handling slowly changing behavior

In [8]:
from jinja2 import Template

scd2_template_update = Template("""

-- Step 1: Expire old records
MERGE INTO {{ target_table }} AS target
USING {{ source_table }} AS source
ON {{ scd_keys }}
AND target.current_flag = true
WHEN MATCHED AND (
    {% for col in tracked_columns %}
        target.{{ col }} != source.{{ col }}{% if not loop.last %} OR {% endif %}
    {% endfor %}
)
THEN UPDATE SET
    current_flag = false,
    effective_end_date = current_date();
""")

scd2_template_insert = Template("""

-- Step 2: Insert new/changed records
INSERT INTO {{ target_table }} (
    {{ insert_columns | join(', ') }},
    effective_start_date,
    effective_end_date,
    current_flag
)
SELECT
    {% for col in insert_columns %}
        source.{{ col }}{% if not loop.last %}, {% endif %}
    {% endfor %},
    current_date(),
    NULL,
    true
FROM {{ source_table }} AS source
LEFT JOIN {{ target_table }} AS target
ON {{ scd_keys }}
AND target.current_flag = true
WHERE
    target.customer_id IS NULL OR
    {% for col in tracked_columns %}
        target.{{ col }} != source.{{ col }}{% if not loop.last %} OR {% endif %}
    {% endfor %};

""")

def run_scd2_merge(source_table, target_table, scd_keys, tracked_columns, insert_columns):
    sql = scd2_template_update.render(
        source_table=source_table,
        target_table=target_table,
        scd_keys=scd_keys,
        tracked_columns=tracked_columns,
        insert_columns=insert_columns
    )
    #print("Executing Update SQL:\n", sql)
    spark.sql(sql)
    sql = scd2_template_insert.render(
        source_table=source_table,
        target_table=target_table,
        scd_keys=scd_keys,
        tracked_columns=tracked_columns,
        insert_columns=insert_columns
    )
    #print("Executing Insert SQL:\n", sql)
    spark.sql(sql)

## Initial data load int empty dim_customer table

In [9]:
from datetime import datetime
from pyspark.sql.functions import lit, current_date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType

# define Schema
schema = StructType([StructField("customer_id", IntegerType(), True),\
                     StructField("name", StringType(), True),\
                     StructField("email", StringType(), True),\
                     StructField("status", StringType(), True),\
                     StructField("effective_start_date", DateType(), True),\
                     StructField("effective_end_date", DateType(), True),\
                     StructField("current_flag", BooleanType(), True)\
                    ])

# Create historical target table
target_df = spark.createDataFrame([
    (1, "Alice", "alice@example.com", "active", datetime(2025, 1, 1), None, True),
    (2, "Bob", "bob@example.com", "active", datetime(2025, 1, 1), None, True),
], schema)

target_df.show()
target_df.write.mode("overwrite").format("delta").saveAsTable("lake.scd.dim_customer")


+-----------+-----+-----------------+------+--------------------+------------------+------------+
|customer_id| name|            email|status|effective_start_date|effective_end_date|current_flag|
+-----------+-----+-----------------+------+--------------------+------------------+------------+
|          1|Alice|alice@example.com|active|          2025-01-01|              NULL|        true|
|          2|  Bob|  bob@example.com|active|          2025-01-01|              NULL|        true|
+-----------+-----+-----------------+------+--------------------+------------------+------------+



## New data including updates and inserts

In [10]:
# Simulating changes to Alice's status and adding a new customer (Charlie)
schema = StructType([StructField("customer_id", IntegerType(), True),\
                     StructField("name", StringType(), True),\
                     StructField("email", StringType(), True),\
                     StructField("status", StringType(), True)\
                    ])
source_df_changed = spark.createDataFrame([
    (1, "Alice", "alice@example.com", "inactive"),  # status changed
    (2, "Bob", "bob@example.com", "inactive"),      # same as before
    (3, "Charlie", "charlie@example.com", "active"),# new record
], schema)

source_df_changed.createOrReplaceTempView("staging_customer")


## Perform the SCD 2 load

Capture history when the following changes
 - name
 - email
 - status


In [11]:
run_scd2_merge(
    source_table="staging_customer",
    target_table="lake.scd.dim_customer",
    scd_keys="target.customer_id = source.customer_id",
    tracked_columns=["name", "email", "status"],
    insert_columns=["customer_id", "name", "email", "status"]
)


Executing Update SQL:
 

-- Step 1: Expire old records
MERGE INTO lake.scd.dim_customer AS target
USING staging_customer AS source
ON target.customer_id = source.customer_id
AND target.current_flag = true
WHEN MATCHED AND (
    
        target.name != source.name OR 
    
        target.email != source.email OR 
    
        target.status != source.status
    
)
THEN UPDATE SET
    current_flag = false,
    effective_end_date = current_date();


Executing Insert SQL:
 

-- Step 2: Insert new/changed records
INSERT INTO lake.scd.dim_customer (
    customer_id, name, email, status,
    effective_start_date,
    effective_end_date,
    current_flag
)
SELECT
    
        source.customer_id, 
    
        source.name, 
    
        source.email, 
    
        source.status
    ,
    current_date(),
    NULL,
    true
FROM staging_customer AS source
LEFT JOIN lake.scd.dim_customer AS target
ON target.customer_id = source.customer_id
AND target.current_flag = true
WHERE
    target.customer_id IS NULL OR
    
        target.name != source.name OR 
    
        target.email != source.email OR 
    
        target.status != source.status
    ;



In [12]:
spark.sql("select * from lake.scd.dim_customer ORDER BY customer_id, effective_start_date").show()

+-----------+-------+-------------------+--------+--------------------+------------------+------------+
|customer_id|   name|              email|  status|effective_start_date|effective_end_date|current_flag|
+-----------+-------+-------------------+--------+--------------------+------------------+------------+
|          1|  Alice|  alice@example.com|  active|          2025-01-01|        2026-08-03|       false|
|          1|  Alice|  alice@example.com|inactive|          2026-08-03|              NULL|        true|
|          2|    Bob|    bob@example.com|  active|          2025-01-01|        2026-08-03|       false|
|          2|    Bob|    bob@example.com|inactive|          2026-08-03|              NULL|        true|
|          3|Charlie|charlie@example.com|  active|          2026-08-03|              NULL|        true|
+-----------+-------+-------------------+--------+--------------------+------------------+------------+

